
# 03 Coordinate Reference Systems & PyProj: Making "Area" Actually Mean Something

This is the notebook that fixes the thing you've probably already noticed feels wrong in notebooks 01
and 02: `.area` on a polygon built from lat/lon coordinates gives you a meaningless number, not square
meters. For a land banking product, where parcel size is a real financial and legal fact, getting this
wrong isn't a cosmetic bug it's the kind of error that undermines the entire trust proposition you're
building.

## The core problem, in one sentence

**Latitude/longitude degrees are not a flat grid of equal-sized squares** a degree of longitude covers
less real distance near the poles than at the equator, and even near the equator (where Cameroon sits),
using raw degrees for area/distance math gives you numbers with no real-world unit at all.

## Two kinds of coordinate reference system (CRS)

| Type | What it is | Example | Good for |
|---|---|---|---|
| **Geographic CRS** | Coordinates as angles (lat/lon) on a model of the curved Earth | `EPSG:4326` (WGS 84 standard GPS) | Storing/exchanging location data, mapping, geocoding |
| **Projected CRS** | Coordinates flattened onto a 2D plane, in real linear units (usually meters) | UTM zones (e.g. `EPSG:32632` for UTM Zone 32N, which covers Cameroon's Douala/Buea area) | **Accurate area, distance, and buffer calculations** |

**The rule that matters in practice: use a geographic CRS (like EPSG:4326) for storing and displaying
data, and reproject to an appropriate projected CRS before doing any area, distance, or buffer
calculation.** Getting this order backwards is the single most common geospatial bug beginners hit.

## Finding the right projected CRS for Cameroon

Cameroon spans two UTM zones: most of the country, including Douala and Buea, falls in **UTM Zone 32N**
(`EPSG:32632`); the far eastern part of the country falls in Zone 33N (`EPSG:32633`). For a product
operating primarily in Douala/Buea, `EPSG:32632` is the projected CRS you'll reach for whenever you need
real measurements.


In [1]:

import geopandas as gpd
from shapely.geometry import Polygon

# The same parcel_a from notebook 01, now inside a GeoDataFrame with an explicit geographic CRS
parcel = gpd.GeoDataFrame(
    {"parcel_id": ["PARC-001"]},
    geometry=[Polygon([(9.240, 4.150), (9.241, 4.150), (9.241, 4.151), (9.240, 4.151)])],
    crs="EPSG:4326",  # WGS 84 standard lat/lon
)

print("Area in EPSG:4326 (meaningless unit square degrees):", parcel.geometry.area.iloc[0])


Area in EPSG:4326 (meaningless unit square degrees): 9.999999999988916e-07


C:\Users\GROUP\AppData\Local\Temp\ipykernel_10928\731721835.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print("Area in EPSG:4326 (meaningless unit square degrees):", parcel.geometry.area.iloc[0])


In [2]:

# Reproject to the correct UTM zone for Cameroon before measuring anything
parcel_projected = parcel.to_crs("EPSG:32632")

area_m2 = parcel_projected.geometry.area.iloc[0]
area_hectares = area_m2 / 10_000

print(f"Area: {area_m2:,.1f} square meters")
print(f"Area: {area_hectares:.4f} hectares")


Area: 12,268.0 square meters
Area: 1.2268 hectares



That's the actual, real-world size of the parcel a number you could put in a due diligence report
with confidence, unlike the raw-degree figure from before.

## `.to_crs()` the one method that does almost everything you need

`GeoDataFrame.to_crs(target_crs)` reprojects every geometry in the GeoDataFrame. That's 90% of what
you'll use PyProj for, through GeoPandas' interface. You rarely need to call PyProj directly but it's
worth seeing what's happening one level down, since GeoPandas is calling PyProj for you under the hood.


In [4]:

from pyproj import Transformer

# The direct PyProj way to convert a single coordinate pair, same transformation as above
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32632", always_xy=True)
x, y = transformer.transform(9.240, 4.150)  # note: always_xy=True means (longitude, latitude) order in, (x, y) out
print(f"Projected coordinates: x={x:.1f}, y={y:.1f} (meters, in the UTM 32N grid)")


Projected coordinates: x=526636.5, y=458711.8 (meters, in the UTM 32N grid)



## Why this matters for buffer distances too

Remember `.buffer(0.003)` from notebook 01? That "0.003" was in degrees a distance that means
something completely different depending on where on Earth you are. If you want "buffer this road by
200 meters," you need to be working in a projected CRS *before* you buffer, or your buffer radius is
scientifically meaningless.


In [5]:

from shapely.geometry import LineString

road = gpd.GeoDataFrame(
    {"name": ["Proposed ring road extension"]},
    geometry=[LineString([(9.235, 4.148), (9.245, 4.152)])],
    crs="EPSG:4326",
)

# WRONG buffering in degrees, distance is meaningless
wrong_buffer = road.copy()
wrong_buffer["geometry"] = wrong_buffer.buffer(0.003)

# RIGHT reproject first, then buffer in real meters, then reproject back if you need to store/display in EPSG:4326
road_projected = road.to_crs("EPSG:32632")
correct_buffer_projected = road_projected.buffer(200)  # a real 200-meter buffer
correct_buffer_back_to_4326 = correct_buffer_projected.to_crs("EPSG:4326")

print("This is the buffer you should actually use for any real distance-based query.")


This is the buffer you should actually use for any real distance-based query.


C:\Users\GROUP\AppData\Local\Temp\ipykernel_10928\4274695178.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  wrong_buffer["geometry"] = wrong_buffer.buffer(0.003)



## Exercises

### Exercise 1
Take the `zones` GeoDataFrame concept from notebook 02 (North Buea / South Buea), reproject it to
`EPSG:32632`, and print each zone's area in hectares.


In [5]:
# Your code here


#### Solution

In [6]:

zones = gpd.GeoDataFrame({
    "zone_name": ["North Buea", "South Buea"],
    "geometry": [
        Polygon([(9.230, 4.145), (9.245, 4.145), (9.245, 4.155), (9.230, 4.155)]),
        Polygon([(9.245, 4.155), (9.260, 4.155), (9.260, 4.165), (9.245, 4.165)]),
    ],
}, crs="EPSG:4326")

zones_projected = zones.to_crs("EPSG:32632")
zones_projected["area_hectares"] = zones_projected.geometry.area / 10_000
zones_projected[["zone_name", "area_hectares"]]


,zone_name,area_hectares
0,North Buea,184.020746
1,South Buea,184.018887



### Exercise 2
Write a function `real_distance_meters(point_a, point_b, source_crs="EPSG:4326")` that takes two Shapely
`Point` objects in lat/lon, reprojects them to `EPSG:32632`, and returns the real distance between them
in meters. (Hint: wrap the points in a tiny GeoSeries to reuse `.to_crs()`, then use `.distance()`.)


In [7]:
# Your code here


#### Solution

In [7]:

from shapely.geometry import Point

def real_distance_meters(point_a, point_b, source_crs="EPSG:4326"):
    gs = gpd.GeoSeries([point_a, point_b], crs=source_crs).to_crs("EPSG:32632")
    return gs.iloc[0].distance(gs.iloc[1])

buea_point = Point(9.24, 4.15)
douala_point = Point(9.71, 4.05)
print(f"Approximate real distance: {real_distance_meters(buea_point, douala_point):,.0f} meters")


Approximate real distance: 53,326 meters



### Exercise 3
Explain in your own words (a markdown cell, no code needed) why using `EPSG:4326` directly for a
"parcels within 200 meters of this road" query would give a wrong answer, even though the query would
run without any error message. This matters because CRS mistakes are silent they don't crash your
code, they just quietly produce wrong numbers, which is worse.


In [9]:
# Write your explanation here as a comment, or switch this cell to Markdown



## What's next

You now have geometry (Shapely), tables of geometry (GeoPandas), and correct real-world measurements
(PyProj/CRS). The next gap: none of your data has come from a real address yet everything so far has
been hand-typed coordinates. **`04_geocoding_with_geopy.ipynb`** covers turning a written address into
a coordinate you can actually plot and query.
